# Aquí veremos distintas formas de comparar mapas, ya sea que provengan de un mismo método de sampleo, o bien, entre mecanismos.

In [1]:
import os
import pandas as pd


ruta_resultados = r"C:\Tesis\Códigos\resultados"


def df_a_labels(df):
    df = df[df["x"] == 1].copy()

    asignacion = dict(zip(df["i"], df["j"]))

    centros = sorted(set(asignacion.values()))
    mapa_centros = {c: idx for idx, c in enumerate(centros)}

    comunas = sorted(asignacion.keys())
    labels = [mapa_centros[asignacion[i]] for i in comunas]

    return labels, comunas


def cargar_labels_desde_resultados(ruta_resultados):
    mapas = {}
    comunas_base = None

    for carpeta in sorted(os.listdir(ruta_resultados)):
        path_carpeta = os.path.join(ruta_resultados, carpeta)

        if not os.path.isdir(path_carpeta):
            continue

        archivo = f"modelo_{carpeta}_asignaciones.csv"
        path_archivo = os.path.join(path_carpeta, archivo)

        if not os.path.exists(path_archivo):
            continue

        df = pd.read_csv(path_archivo)
        labels, comunas = df_a_labels(df)

        if comunas_base is None:
            comunas_base = comunas
        elif comunas != comunas_base:
            raise ValueError(f"Las comunas de {carpeta} no coinciden con las del resto")

        mapas[carpeta] = labels

    return mapas, comunas_base

In [2]:
from funciones_metricas import matriz_metricas, comunas_inestables, estabilidad_comuna, comunas_inestables_nombres

In [3]:
mapas, comunas = cargar_labels_desde_resultados(ruta_resultados)

In [4]:
idx = comunas.index("arica")
mapas["t_001"][idx]

3

In [5]:
matriz_ari = matriz_metricas(mapas, metrica="ari")
matriz_nmi = matriz_metricas(mapas, metrica="nmi")
matriz_vi  = matriz_metricas(mapas, metrica="vi")

In [6]:
matriz_ari

,t_001,t_002,t_003,t_004,t_005,t_006,t_007,t_008,t_009,t_010,...,t_091,t_092,t_093,t_094,t_095,t_096,t_097,t_098,t_099,t_100
t_001,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_002,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_003,0.952613,0.952613,1.000000,0.951446,0.952613,0.952613,0.952613,0.955742,0.952613,0.952613,...,0.951446,0.937738,0.952613,0.952613,0.951446,0.952613,0.951446,0.952613,0.952613,0.952613
t_004,0.995173,0.995173,0.951446,1.000000,0.995173,0.995173,0.995173,0.986678,0.995173,0.995173,...,1.000000,0.974273,0.995173,0.995173,1.000000,0.995173,1.000000,0.995173,0.995173,0.995173
t_005,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
t_096,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_097,0.995173,0.995173,0.951446,1.000000,0.995173,0.995173,0.995173,0.986678,0.995173,0.995173,...,1.000000,0.974273,0.995173,0.995173,1.000000,0.995173,1.000000,0.995173,0.995173,0.995173
t_098,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000
t_099,1.000000,1.000000,0.952613,0.995173,1.000000,1.000000,1.000000,0.987845,1.000000,1.000000,...,0.995173,0.973436,1.000000,1.000000,0.995173,1.000000,0.995173,1.000000,1.000000,1.000000


In [7]:
import numpy as np

In [8]:
def resumen_matriz(M):
    valores = M.values
    valores = valores[~np.eye(valores.shape[0], dtype=bool)]
    return {
        "mean": valores.mean(),
        "std": valores.std(),
        "min": valores.min(),
        "max": valores.max()
    }

resumen_ari = resumen_matriz(matriz_ari)
resumen_nmi = resumen_matriz(matriz_nmi)
resumen_vi  = resumen_matriz(matriz_vi)

In [9]:
resumen_ari

{'mean': np.float64(0.9838675752148585),
 'std': np.float64(0.018408153260988207),
 'min': np.float64(0.9377378752551687),
 'max': np.float64(1.0)}

In [10]:
resumen_nmi

{'mean': np.float64(0.9805550315251441),
 'std': np.float64(0.020290122030424188),
 'min': np.float64(0.941093068221218),
 'max': np.float64(1.0)}

In [11]:
resumen_vi

{'mean': np.float64(0.11951462797250685),
 'std': np.float64(0.12472648059717184),
 'min': np.float64(0.0),
 'max': np.float64(0.36231215404058315)}

In [12]:
inestables = comunas_inestables(mapas)
len(inestables)

207

In [13]:
nombres_inesatables = comunas_inestables_nombres(mapas, comunas)

In [17]:
len(nombres_inesatables)

207

In [14]:
nombres_inesatables

{'alhue': {17: 50, 13: 6, 16: 32, 14: 7, 11: 5},
 'ancud': {15: 62, 14: 38},
 'andacollo': {16: 62, 15: 38},
 'antartica': {22: 73, 21: 27},
 'arauco': {6: 99, 13: 1},
 'aysen': {20: 73, 19: 27},
 'buin': {17: 57, 13: 4, 16: 34, 11: 5},
 'bulnes': {5: 98, 4: 2},
 'cabo de hornos': {22: 73, 21: 27},
 'calbuco': {15: 62, 14: 38},
 'caldera': {7: 96, 23: 4},
 'calera de tango': {4: 98, 16: 2},
 'camina': {10: 96, 9: 4},
 'canela': {8: 96, 7: 4},
 'castro': {15: 62, 14: 38},
 'cerrillos': {13: 81, 11: 6, 12: 13},
 'cerro navia': {13: 78, 4: 2, 21: 11, 14: 7, 5: 2},
 'chaiten': {15: 62, 14: 38},
 'chanaral': {7: 96, 23: 4},
 'chepica': {19: 62, 18: 38},
 'chiguayante': {6: 99, 13: 1},
 'chile chico': {20: 73, 19: 27},
 'chillan': {5: 98, 4: 2},
 'chillan viejo': {5: 98, 4: 2},
 'chimbarongo': {19: 62, 18: 38},
 'chonchi': {15: 62, 14: 38},
 'cisnes': {20: 73, 19: 27},
 'cobquecura': {5: 98, 4: 2},
 'cochamo': {15: 62, 14: 38},
 'cochrane': {20: 73, 19: 27},
 'codegua': {19: 62, 18: 38},
 'c

In [18]:
mapas_estabilidad = estabilidad_comuna(mapas)

In [20]:
len(mapas_estabilidad)

346

In [21]:
mapas_estabilidad

[1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 1.0,
 0.73,
 1.0,
 1.0,
 0.99,
 1.0,
 0.73,
 0.57,
 0.98,
 1.0,
 0.73,
 1.0,
 1.0,
 0.62,
 0.96,
 1.0,
 0.98,
 1.0,
 1.0,
 0.96,
 0.96,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 1.0,
 1.0,
 0.81,
 0.78,
 0.62,
 0.96,
 1.0,
 0.62,
 0.99,
 0.73,
 0.98,
 0.98,
 0.62,
 1.0,
 0.62,
 0.73,
 0.98,
 0.62,
 0.73,
 0.62,
 0.98,
 0.98,
 0.62,
 1.0,
 0.96,
 0.78,
 1.0,
 0.62,
 0.62,
 0.99,
 0.68,
 1.0,
 1.0,
 1.0,
 0.96,
 0.96,
 0.99,
 0.49,
 0.73,
 1.0,
 1.0,
 0.55,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 0.96,
 0.62,
 0.96,
 0.98,
 0.57,
 1.0,
 1.0,
 1.0,
 1.0,
 0.81,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 0.62,
 0.96,
 1.0,
 1.0,
 1.0,
 0.62,
 0.73,
 1.0,
 0.62,
 1.0,
 0.99,
 0.99,
 0.96,
 0.96,
 0.62,
 0.62,
 0.73,
 0.96,
 0.57,
 1.0,
 1.0,
 0.82,
 1.0,
 0.62,
 0.91,
 0.49,
 0.62,
 1.0,
 0.91,
 0.82,
 0.96,
 0.76,
 0.96,
 0.73,
 0.73,
 1.0,
 0.8,
 0.55,
 0.62,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.62,
 0.62,
 1.0,
 0.62,
 0.81,
 0.49,
 0.62,
 1.0,
 1.0,
 1.0,
 1.0

In [22]:
count = 0
for mapa in mapas_estabilidad:
    if mapa == 1.0:
        count += 1
print(f'cantidad de mapas estables: {count}')

cantidad de mapas estables: 139


In [23]:
len(nombres_inesatables) + count 

346

In [7]:
import os
import pandas as pd
import numpy as np

BASE_PATH = "resultados"  # ajusta si es necesario

# comunas con población
comunas_df = pd.read_excel("DataChile/comunas.xlsx")
comunas_df["comuna"] = comunas_df["comuna"].str.lower()

poblacion_dict = dict(zip(comunas_df["comuna"], comunas_df["poblacion2017"]))

In [3]:
def cargar_resultados(ruta):
    factibles_path = os.path.join(ruta, "centros_factibles.txt")
    
    if not os.path.exists(factibles_path):
        return []
    
    resultados = []
    with open(factibles_path, "r", encoding="utf-8") as f:
        for line in f:
            comunas = [c.strip().lower() for c in line.strip().split(",")]
            resultados.append(comunas)
    
    return resultados

In [4]:
def metricas_poblacion(centros):
    pops = [poblacion_dict.get(c, 0) for c in centros]
    
    return {
        "pop_mean": np.mean(pops),
        "pop_std": np.std(pops),
        "pop_min": np.min(pops),
        "pop_max": np.max(pops),
        "pop_cv": np.std(pops) / np.mean(pops) if np.mean(pops) > 0 else 0
    }

In [5]:
def analizar_carpeta(nombre_carpeta):
    ruta = os.path.join(BASE_PATH, nombre_carpeta)
    soluciones = cargar_resultados(ruta)
    
    if len(soluciones) == 0:
        return None
    
    metricas = []
    
    for sol in soluciones:
        m = metricas_poblacion(sol)
        metricas.append(m)
    
    df = pd.DataFrame(metricas)
    
    resumen = df.agg(["mean", "std"])
    resumen["metodo"] = nombre_carpeta
    
    return resumen

In [8]:
carpetas = os.listdir(BASE_PATH)

CL = [c for c in carpetas if "CL" in c]
SL = [c for c in carpetas if "SL" in c]

In [9]:
def construir_tabla(lista_carpetas):
    resultados = []
    
    for c in lista_carpetas:
        resumen = analizar_carpeta(c)
        if resumen is not None:
            resumen["carpeta"] = c
            resultados.append(resumen)
    
    return pd.concat(resultados)

In [10]:
tabla_CL = construir_tabla(CL)
tabla_SL = construir_tabla(SL)

print("=== CL ===")
display(tabla_CL)

print("=== SL ===")
display(tabla_SL)

=== CL ===


,pop_mean,pop_std,pop_min,pop_max,pop_cv,metodo,carpeta
mean,61589.104447,62502.866211,625.0,244296.424528,1.009397,resultados_CL_PIVOTAL_epsilon_0.87,resultados_CL_PIVOTAL_epsilon_0.87
std,5526.229048,11432.812994,0.0,85214.351638,0.118115,resultados_CL_PIVOTAL_epsilon_0.87,resultados_CL_PIVOTAL_epsilon_0.87
mean,61413.419286,64048.197717,625.0,253595.940000,1.036320,resultados_CL_samp_epsilon_0.87,resultados_CL_samp_epsilon_0.87
std,6242.674523,12313.886202,0.0,89360.973224,0.120185,resultados_CL_samp_epsilon_0.87,resultados_CL_samp_epsilon_0.87
mean,56606.824675,59891.242994,625.0,234510.636364,1.054241,resultados_CL_Systematic_epsilon_0.87,resultados_CL_Systematic_epsilon_0.87
std,5915.143597,9471.963299,0.0,67002.908663,0.068970,resultados_CL_Systematic_epsilon_0.87,resultados_CL_Systematic_epsilon_0.87


=== SL ===


,pop_mean,pop_std,pop_min,pop_max,pop_cv,metodo,carpeta
mean,122362.708534,143420.747299,1364.941450,555637.200000,1.183951,resultados_SL_piv_epsilon_0.87,resultados_SL_piv_epsilon_0.87
std,16584.961532,12639.554037,1893.999381,32325.092620,0.118064,resultados_SL_piv_epsilon_0.87,resultados_SL_piv_epsilon_0.87
mean,119925.050575,141798.200895,1067.899380,555875.320000,1.201627,resultados_SL_samp_epsilon_0.87,resultados_SL_samp_epsilon_0.87
std,18658.545979,13187.865937,1395.916945,27151.589792,0.155279,resultados_SL_samp_epsilon_0.87,resultados_SL_samp_epsilon_0.87
mean,112062.764356,140847.055254,408.631960,552777.830000,1.268025,resultados_SL_sys_epsilon_0.87,resultados_SL_sys_epsilon_0.87
std,12866.060310,8060.375908,465.330026,21950.941002,0.111258,resultados_SL_sys_epsilon_0.87,resultados_SL_sys_epsilon_0.87


In [11]:
def tabla_simple(tabla):
    return tabla.loc["mean"][[
        "pop_mean", "pop_std", "pop_min", "pop_max", "pop_cv"
    ]]

print("CL")
display(tabla_simple(tabla_CL))

print("SL")
display(tabla_simple(tabla_SL))

CL


,pop_mean,pop_std,pop_min,pop_max,pop_cv
mean,61589.104447,62502.866211,625.0,244296.424528,1.009397
mean,61413.419286,64048.197717,625.0,253595.940000,1.036320
mean,56606.824675,59891.242994,625.0,234510.636364,1.054241


SL


,pop_mean,pop_std,pop_min,pop_max,pop_cv
mean,122362.708534,143420.747299,1364.94145,555637.20,1.183951
mean,119925.050575,141798.200895,1067.89938,555875.32,1.201627
mean,112062.764356,140847.055254,408.63196,552777.83,1.268025
